# 🔥 Sıfırdan PyTorch → əl yazısı rəqəmləri tanıyırıq (MNIST)
### Modul 4 · Dərs 2

Keçən dərsdə **TensorFlow Playground**-da oynadıq: neyronlar, layer-lər, çəkilər, training. Hamısını mausla edirdik.
Bu gün eyni şeyi **kodla**, **PyTorch** kitabxanası ilə edəcəyik.

Dərsin sonunda əlimizdə belə bir proqram olacaq: **mausla rəqəm çəkirsən**, neyron şəbəkə də tapır ki, bu hansı rəqəmdir. 🎯

| Addım | Mövzu | Vaxt |
|---|---|---|
| 1 | Tensor nədir, necə yaradılır | 15 dəq |
| 2 | Əməliyyatlar, shape, broadcasting | 15 dəq |
| 3 | Autograd: törəmələri PyTorch özü hesablayır | 15 dəq |
| 4 | Gradient descent əllə → sonra eyni şey `nn` ilə | 20 dəq |
| 5 | MNIST dataseti, DataLoader | 15 dəq |
| 6 | İlk şəbəkə (MLP): training, accuracy | 20 dəq |
| 7 | Konvolyusiya şəbəkəsi (CNN), modeli yadda saxlamaq | 20 dəq |
| 8 | Qt-də proqram: rəqəm çəkirik | 15 dəq |

## 0️⃣ Hazırlıq

Lazım olan kitabxanalar (terminalda bir dəfə yazırıq):

```bash
pip install torch torchvision matplotlib numpy PySide6
```

- `torch` — PyTorch-un özü
- `torchvision` — şəkillər və hazır datasetlər (MNIST də burdadır)
- `matplotlib` — qrafik çəkmək üçün
- `PySide6` — Python üçün Qt. Rəqəm çəkmək üçün pəncərəni bununla quracağıq

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch versiyası:", torch.__version__)

PyTorch versiyası: 2.15.0


---
# 🅰️ Hissə 1. PyTorch necə işləyir

## 1️⃣ Tensor — əsas obyektimiz

**Tensor** sadəcə rəqəmlərdən ibarət massivdir. Python-dakı list və ya NumPy array kimi bir şey.

| Neçə ölçü | Adı | Nümunə |
|---|---|---|
| 0 | skalyar | temperatur: `23.5` |
| 1 | vektor | tələbənin qiymətləri: `[5, 4, 5]` |
| 2 | matris | Excel cədvəli, ağ-qara şəkil |
| 3 | 3D tensor | rəngli şəkil (RGB × hündürlük × en) |
| 4 | 4D tensor | bir **dəstə** şəkil |

Neyron şəbəkəyə tensor girir, içəridə tensorlar bir-birinə vurulur, çıxışda da yenə tensor alırıq. Hər şey tensordur.

In [2]:
a = torch.tensor([1, 2, 3])              # list-dən
b = torch.tensor([[1., 2.], [3., 4.]])   # 2×2 matris

print(a)
print(b)

tensor([1, 2, 3])
tensor([[1., 2.],
        [3., 4.]])


Hər tensorun üç vacib xüsusiyyəti var:

- `shape` — **forması**: hər ölçü üzrə neçə element var
- `dtype` — **rəqəmlərin tipi**: tam (`int64`) və ya kəsr (`float32`)
- `device` — **harada saxlanılır**: adi yaddaşda (`cpu`) və ya videokartda (`cuda`, Mac-də `mps`)

In [3]:
print("a:", a.shape, a.dtype, a.device)
print("b:", b.shape, b.dtype, b.device)

a: torch.Size([3]) torch.int64 cpu
b: torch.Size([2, 2]) torch.float32 cpu


> 💡 Diqqət edin: `a` tam rəqəmlərdir (`int64`), `b` isə kəsr (`float32`), çünki `1.` yazmışıq, nöqtə ilə.
> Neyron şəbəkələr **float32** ilə işləyir. `int` versəniz, PyTorch çox vaxt error verəcək.

### Hazır tensor "şablonları"

In [4]:
z = torch.zeros(2, 3)     # sıfırlar, shape (2, 3)
o = torch.ones(2, 3)      # birlər
r = torch.rand(2, 3)      # 0 ilə 1 arası random rəqəmlər (bərabər paylanma)
n = torch.randn(2, 3)     # normal paylanmadan random (orta 0, std 1)
s = torch.arange(0, 10)   # [0, 1, ..., 9]

print("zeros:\n", z)
print("rand:\n", r)
print("randn:\n", n)
print("arange:", s)

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
rand:
 tensor([[0.3867, 0.8494, 0.0431],
        [0.6637, 0.1784, 0.1385]])
randn:
 tensor([[ 0.1804,  0.1390,  0.6202],
        [-1.6623, -0.4268,  0.0430]])
arange: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


> 🎲 `torch.rand` hər dəfə başqa rəqəmlər verir. Sinifdə hamıda eyni nəticə çıxsın deyə random-un
> "toxumunu" sabitləyirik: `torch.manual_seed(42)`.

In [5]:
torch.manual_seed(42)
print(torch.rand(3))
torch.manual_seed(42)
print(torch.rand(3))   # eyni rəqəmlər!

tensor([0.8823, 0.9150, 0.3829])
tensor([0.8823, 0.9150, 0.3829])


### ✍️ Mini tapşırıq 1
1-dən 12-yə qədər 12 rəqəmdən ibarət tensor yaradın, onun `shape` və `dtype`-ını çap edin.

In [ ]:
# kodunuzu bura yazın

## 2️⃣ Tensorlarla əməliyyatlar

### Element-element əməliyyatlar
`+ - * /` **element-element** işləyir: eyni yerdə duran elementlər bir-biri ilə.

In [ ]:
x = torch.tensor([1., 2., 3.])
y = torch.tensor([10., 20., 30.])

print("x + y =", x + y)
print("x * y =", x * y)      # matris vurması DEYİL, element-element vurmadır
print("x ** 2 =", x ** 2)
print("sum:", x.sum(), " mean:", x.mean(), " max:", x.max())

### Matris vurması `@`

Neyron şəbəkənin əsas əməliyyatı budur. Neyronlardan ibarət bir layer: **çıxış = giriş @ W + b**.

Shape qaydası: `(m, k) @ (k, n) → (m, n)`. Yəni "içəridəki" ölçülər üst-üstə düşməlidir.

In [ ]:
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])      # (2, 3)
B = torch.tensor([[1., 0.],
                  [0., 1.],
                  [1., 1.]])          # (3, 2)

C = A @ B                              # (2, 3) @ (3, 2) → (2, 2)
print(C)
print(C.shape)

### Broadcasting — "uzatma"

Shape-lər fərqli olanda PyTorch bəzən kiçik tensoru özü "uzadır". Məsələn, vektoru matrisin hər sətrinə əlavə etmək:

In [ ]:
M = torch.zeros(3, 4)
row = torch.tensor([1., 2., 3., 4.])   # (4,)

print(M + row)    # row 3 sətrin HƏR BİRİNƏ əlavə olundu

Layer-də `b` (bias) də məhz belə əlavə olunur: bir sətir bias dəstədəki bütün nümunələrə gedir.

### Shape-i dəyişmək: `reshape`, `flatten`

MNIST şəkli 28×28 = **784** pikseldir. Sadə şəbəkə girişdə **vektor** istəyir, ona görə də şəkli bir sətrə "açırıq".

In [ ]:
img = torch.arange(16).reshape(4, 4)   # 4×4 "şəkil"
print(img)

flat = img.reshape(-1)                 # -1 = "özün hesabla"
print(flat, flat.shape)

back = flat.reshape(2, 8)
print(back)

### İndeksləmə — NumPy-dakı kimi

In [ ]:
print(img[0])        # birinci sətir
print(img[:, 0])     # birinci sütun
print(img[1:3, 1:3]) # ortadan 2×2 parça
print(img[img > 10]) # yalnız 10-dan böyük elementlər

### NumPy ↔ PyTorch

Biri o birinə rahatca çevrilir. Proqramın pəncərəsindən şəkli götürəndə bu bizə lazım olacaq.

In [ ]:
np_arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(np_arr)    # numpy → torch
print(t)

back_to_np = t.numpy()          # torch → numpy
print(back_to_np, type(back_to_np))

### Device: CPU, yoxsa GPU

Apple çipli Mac-lərdə (M1/M2/M3…) **MPS** sürətləndiricisi var. NVIDIA-lı PC-də **CUDA**. Heç biri yoxdursa, adi **CPU**.
Tensorlar və model **eyni device-da** olmalıdır, yoxsa error alacaqsınız.

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Hesablayacağımız yer:", device)

t = torch.ones(3).to(device)
print(t, t.device)

### ✍️ Mini tapşırıq 2
1. `(5, 3)` shape-li random `X` matrisi və `(3, 2)` shape-li `W` matrisi yaradın.
2. `X @ W` hesablayın. Hansı shape alındı? Niyə?
3. İndi `W @ X` yoxlayın. Nə oldu?

In [ ]:
# kodunuzu bura yazın

## 3️⃣ Autograd — törəmələri PyTorch özü hesablayır

Playground-u xatırlayaq: şəbəkə çəkiləri **error-u azaldan tərəfə** sürüşdürərək öyrənir.
Hansı tərəfə sürüşdürmək lazım olduğunu bilmək üçün **törəmə** (gradient) lazımdır: "çəkini bir az artırsam, error artacaq, yoxsa azalacaq?"

Milyon çəki üçün törəməni əllə hesablamaq mümkün deyil. PyTorch bunu **özü** edir:

1. tensoru `requires_grad=True` ilə işarələyirik — "bunu izlə"
2. ondan nəsə hesablayırıq
3. `.backward()` çağırırıq
4. törəmə `.grad`-ın içində olur

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2          # y = x²
y.backward()        # dy/dx hesabla

print("y =", y.item())
print("dy/dx =", x.grad.item())   # x²-nin törəməsi 2x → 2·3 = 6

✅ Məktəb riyaziyyatı: $(x^2)' = 2x$, $x=3$ olanda $6$ alırıq. PyTorch da eyni şeyi hesabladı.

Bir az çətin nümunə. PyTorch istənilən düsturun öhdəsindən gəlir:

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

x = torch.tensor(4.0)          # giriş (onu izləmirik)
y_true = torch.tensor(15.0)    # düzgün cavab

y_pred = w * x + b             # 2·4 + 1 = 9
loss = (y_pred - y_true) ** 2  # (9 - 15)² = 36

loss.backward()

print("loss =", loss.item())
print("d loss / d w =", w.grad.item())   # mənfidir → w-ni ARTIRMAQ lazımdır
print("d loss / d b =", b.grad.item())

> 💡 Gradient **mənfidir** → `w`-ni artırsaq, error azalacaq. Deməli, çəkini gradientin **əksinə** sürüşdürürük:
>
> $$w_{yeni} = w - lr \cdot \frac{\partial loss}{\partial w}$$
>
> `lr` (learning rate) addımın ölçüsüdür. Bax bu, **gradient descent**-dir.

## 4️⃣ İlk "modeli" əllə öyrədirik

Məsələ belədir: nöqtələrimiz var, təxminən $y = 2x + 1$ xəttinin üstündədir. Model bunu bilmir.
Qoy özü tapsın ki, `w ≈ 2`, `b ≈ 1`.

In [ ]:
torch.manual_seed(0)
X = torch.linspace(-1, 1, 50)
Y = 2 * X + 1 + 0.2 * torch.randn(50)   # xətt + bir az səs-küy (noise)

plt.scatter(X, Y)
plt.title("Datamız")
plt.show()

In [ ]:
w = torch.tensor(0.0, requires_grad=True)   # sıfırdan başlayırıq
b = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(100):
    # 1. proqnoz
    Y_pred = w * X + b
    # 2. error (MSE)
    loss = ((Y_pred - Y) ** 2).mean()
    # 3. gradientlər
    loss.backward()
    # 4. gradient descent addımı
    with torch.no_grad():            # burda əməliyyatları izləməyə ehtiyac yoxdur
        w -= lr * w.grad
        b -= lr * b.grad
    # 5. gradientləri sıfırla (yoxsa üst-üstə yığılır!)
    w.grad.zero_()
    b.grad.zero_()

    if step % 20 == 0:
        print(f"addım {step:3d}: loss={loss.item():.4f}  w={w.item():.3f}  b={b.item():.3f}")

print(f"\nNəticə: w={w.item():.3f}, b={b.item():.3f}")

In [ ]:
plt.scatter(X, Y, label="data")
plt.plot(X, (w * X + b).detach(), color="red", label="model")
plt.legend()
plt.show()

🎉 Model `w ≈ 2`, `b ≈ 1`-i özü tapdı.

Bu **5 addımı** yadda saxlayın. Ən böyük neyron şəbəkə də daxil olmaqla, **hər** training-də olacaqlar:

1. **forward** — proqnoz
2. **loss** — nə qədər səhv etdik
3. **backward** — gradientlər
4. **step** — çəkiləri sürüşdür
5. **zero_grad** — gradientləri sıfırla

### Eyni şey, amma "böyüklər kimi": `nn` və `optim`

Çəkilər milyonlarla olanda `w`, `b`-ni əllə yazmaq olmur. PyTorch-da hazır "kərpiclər" var:

- `nn.Linear(giriş, çıxış)` — neyronlardan ibarət layer (`W` və `b`-ni içində özü saxlayır)
- `nn.MSELoss()` — hazır loss funksiyası
- `torch.optim.SGD` — hazır gradient descent

In [ ]:
import torch.nn as nn

model = nn.Linear(1, 1)                       # 1 giriş, 1 çıxış: y = w·x + b
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

X2 = X.reshape(-1, 1)    # nn.Linear belə shape istəyir: (nümunələrin sayı, feature-ların sayı)
Y2 = Y.reshape(-1, 1)

for step in range(100):
    Y_pred = model(X2)            # 1. forward
    loss = loss_fn(Y_pred, Y2)    # 2. loss
    optimizer.zero_grad()         # 5. sıfırla (adətən backward-dan əvvəl edirlər)
    loss.backward()               # 3. backward
    optimizer.step()              # 4. step

print("w =", model.weight.item(), " b =", model.bias.item())

Nəticə eynidir, amma kod qısadır və çəkilərin sayından **asılı deyil**.

### Öz neyron şəbəkəmiz: `nn.Module` class-ı

Əsl şəbəkə bir neçə ardıcıl layer-dir, aralarında da **aktivasiya funksiyası** olur (Playground-dakı kimi: ReLU, Tanh…).
Aktivasiya olmasa, bir neçə linear layer elə bir linear layer-ə bərabərdir, şəbəkə də mürəkkəb heç nə öyrənə bilmir.

Şəbəkəni class kimi yazırlar:
- `__init__`-də layer-ləri yaradırıq
- `forward`-da datanın onlardan necə keçdiyini yazırıq

In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 8)    # 2 giriş → 8 neyron
        self.layer2 = nn.Linear(8, 1)    # 8 → 1 çıxış

    def forward(self, x):
        x = self.layer1(x)
        x = torch.relu(x)                # ReLU aktivasiyası: max(0, x)
        x = self.layer2(x)
        return x


net = TinyNet()
print(net)

fake_input = torch.rand(5, 2)            # 5 nümunə, hərəsində 2 feature
print("çıxış:", net(fake_input).shape)   # (5, 1)

n_params = sum(p.numel() for p in net.parameters())
print("cəmi çəki:", n_params)            # 2·8+8 + 8·1+1 = 33

### Şəbəkə class-ı necə seçir: logits → softmax

Rəqəmlər üçün şəbəkənin **10 çıxışı** olacaq: 0…9 rəqəmlərinin hər biri üçün biri. Xam çıxışlara **logits** deyirlər, istənilən rəqəm ola bilər.
`softmax` onları **ehtimallara** çevirir (hamısı 0 ilə 1 arası, cəmi = 1).

Classification üçün loss funksiyası `nn.CrossEntropyLoss`-dur. Softmax-ı içində özü edir, ona görə şəbəkə sadəcə logits qaytarır.

In [ ]:
logits = torch.tensor([2.0, 0.5, -1.0])     # şəbəkə 3 class haqqında "düşünür"
probs = torch.softmax(logits, dim=0)
print("ehtimallar:", probs, " cəmi:", probs.sum().item())
print("şəbəkənin cavabı: class", probs.argmax().item())

loss_fn = nn.CrossEntropyLoss()
print("düzgün class 0 olsa, loss:", loss_fn(logits.unsqueeze(0), torch.tensor([0])).item())
print("düzgün class 2 olsa, loss:", loss_fn(logits.unsqueeze(0), torch.tensor([2])).item())

> 💡 Şəbəkə düzgün cavabdan əmindirsə, loss kiçik olur. Səhv cavabdan əmindirsə, loss böyük olur.

---
# 🅱️ Hissə 2. MNIST — əl yazısı rəqəmlər

**MNIST** əl ilə yazılmış rəqəmlərin məşhur datasetidir, içində **70 000** şəkil var:
- 60 000-i öyrətmək üçün (train)
- 10 000-i yoxlamaq üçün (test)

Hər şəkil **28×28 piksel**, ağ-qaradır. Rəqəm qara fonda ağ rənglə yazılıb.

## 5️⃣ Datanı yükləyirik

`torchvision` MNIST-i özü yükləyəcək (təxminən 12 MB, yalnız ilk dəfə).

`transforms` — yükləyəndə hər şəkillə nə edək:
- `ToTensor()` — şəkli `(1, 28, 28)` shape-li tensora çevirir, piksellər 0…255-dən 0…1-ə keçir
- `Normalize(0.1307, 0.3081)` — datanı ortası 0, yayılması 1 olacaq şəkildə sürüşdürürük (şəbəkə belə daha tez öyrənir). Rəqəmlər bütün MNIST-in mean və std-sidir.

In [ ]:
from torchvision import datasets, transforms

MEAN, STD = 0.1307, 0.3081

basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_data = datasets.MNIST(root="data", train=True, download=True, transform=basic_transform)
test_data = datasets.MNIST(root="data", train=False, download=True, transform=basic_transform)

print("train:", len(train_data), " test:", len(test_data))

In [ ]:
image, label = train_data[0]
print("şəklin shape-i:", image.shape)   # (1, 28, 28): 1 kanal (ağ-qara), 28×28
print("cavab:", label)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    img, lbl = train_data[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(lbl)
    ax.axis("off")
plt.show()

Bir şəklə **rəqəmlər kimi** baxaq. Kompüter məhz bunu görür:

In [ ]:
raw = train_data.data[0]           # orijinal piksellər 0..255, transform-suz
print(raw[4:24, 4:24])             # ortadakı hissə

## DataLoader — datanı dəstə-dəstə veririk

Şəbəkəyə şəkilləri bir-bir vermək yavaşdır. 60 000-i birdən vermək isə yaddaşa sığmaz.
Ona görə 64 şəkillik **batch**-lər götürürük.

`DataLoader`:
- dataseti batch-lərə bölür
- `shuffle=True` — hər epoch-da qarışdırır (şəbəkə sıranı əzbərləməsin deyə)

**Epoch** — bütün 60 000 şəkildən bir dəfə tam keçmək.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

images, labels = next(iter(train_loader))
print("şəkil batch-i:", images.shape)   # (64, 1, 28, 28)
print("cavab batch-i:", labels.shape)   # (64,)
print("bir epoch-da batch sayı:", len(train_loader))

## 6️⃣ Rəqəmlər üçün ilk şəbəkə: MLP

Ən sadə variant Playground-dakı kimidir, sadəcə daha böyük:

```
şəkil 28×28 → 784 rəqəmə aç → [128 neyron] → ReLU → [64 neyron] → ReLU → [10 çıxış]
```

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()          # (B, 1, 28, 28) → (B, 784)
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)                   # logits, softmax lazım deyil


mlp = MLP().to(device)
print(mlp)
print("çəki sayı:", sum(p.numel() for p in mlp.parameters()))

Training-i və yoxlamanı funksiya kimi yazaq, ikinci şəbəkə üçün də lazım olacaq.
İçində yenə həmin **5 addım** var, sadəcə batch-lər üzrə.

> `model.train()` / `model.eval()` — rejimi dəyişir (Dropout kimi layer-lər üçün vacibdir, aşağıda görəcəyik).
> Yoxlayanda `torch.no_grad()` — gradientlər lazım deyil, belə daha sürətlidir.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)              # 1. forward
        loss = loss_fn(logits, labels)      # 2. loss
        optimizer.zero_grad()               # 5. zero_grad
        loss.backward()                     # 3. backward
        optimizer.step()                    # 4. step

        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)   # ən böyük çıxışın nömrəsi
            correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)


print("Training-DƏN ƏVVƏL accuracy:", evaluate(mlp, test_loader))   # ~10%, sadəcə təxmin edir

`SGD` əvəzinə **Adam** götürək. Bu, təkmilləşdirilmiş gradient descent-dir, hər çəki üçün addımın ölçüsünü özü seçir. Demək olar ki, həmişə daha tez öyrənir.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)

for epoch in range(1, 4):
    loss = train_one_epoch(mlp, train_loader, optimizer, loss_fn)
    acc = evaluate(mlp, test_loader)
    print(f"epoch {epoch}: loss={loss:.4f}  test accuracy={acc:.2%}")

🎉 Şəbəkənin **heç vaxt görmədiyi** şəkillərdə təxminən **97%** düzgün cavab!

### Şəbəkə harda səhv edir, baxaq

In [ ]:
def show_mistakes(model, n=12):
    model.eval()
    images, labels = next(iter(test_loader))
    with torch.no_grad():
        preds = model(images.to(device)).argmax(dim=1).cpu()
    wrong = (preds != labels).nonzero().flatten()[:n]

    fig, axes = plt.subplots(1, len(wrong), figsize=(1.5 * len(wrong), 2))
    for ax, i in zip(axes, wrong):
        ax.imshow(images[i].squeeze(), cmap="gray")
        ax.set_title(f"şəbəkə: {preds[i]}\nəslində: {labels[i]}", fontsize=9)
        ax.axis("off")
    plt.show()


show_mistakes(mlp)

Səhvlərin çoxu başa düşüləndir. Belə rəqəmləri insan da çətinliklə oxuyar.

### MLP-nin problemi
MLP şəkli bir sətrə açır və **"qonşu piksellər" anlayışını itirir**. Onun üçün şəkil sadəcə 784 ayrı rəqəmdir.
Rəqəmi bir-iki piksel sürüşdürsən, MLP üçün bu artıq tamam başqa rəqəmlərdir. Biz isə mausla çəkəndə rəqəmlər
əyri, qalın, kənara sürüşmüş olacaq… Bizə **formanı** başa düşən şəbəkə lazımdır.

## 7️⃣ Konvolyusiya şəbəkəsi (CNN)

**Konvolyusiyanın** ideyası: 3×3-lük kiçik pəncərə (filter) şəklin üstündə "sürüşür" və sadə bir naxış axtarır, məsələn şaquli xətt və ya künc.
Eyni filter şəklin **istənilən yerində** işləyir, ona görə rəqəm sürüşsə də problem olmur.

```
şəkil 1×28×28
  → Conv 3×3, 32 filter → ReLU → MaxPool (2 dəfə kiçildirik)  → 32×14×14   "xətlər, künclər"
  → Conv 3×3, 64 filter → ReLU → MaxPool                       → 64×7×7    "ilməklər, qövslər"
  → aç → Linear 128 → ReLU → Dropout → Linear 10               → cavab
```

- **MaxPool** — hər 2×2 kvadratdan ən böyük rəqəmi saxlayırıq. Şəkil kiçilir, əsas məlumat qalır.
- **Dropout** — training zamanı neyronların 25%-ni random "söndürür". Şəbəkə bir neyrona güvənə bilmir → overfitting azalır.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # (B, 32, 14, 14)
        x = self.pool(torch.relu(self.conv2(x)))   # (B, 64, 7, 7)
        x = torch.flatten(x, 1)                    # (B, 3136)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)


cnn = Net().to(device)
print(cnn)
print("çəki sayı:", sum(p.numel() for p in cnn.parameters()))

Bir batch üzərində shape-ləri yoxlayaq. Faydalı vərdişdir, çünki ən çox rast gəlinən error-lar shape-lərlə bağlıdır:

In [ ]:
x = images[:4].to(device)
print("giriş:       ", x.shape)
x = cnn.pool(torch.relu(cnn.conv1(x))); print("conv1-dən sonra:", x.shape)
x = cnn.pool(torch.relu(cnn.conv2(x))); print("conv2-dən sonra:", x.shape)

### Augmentation: "xarab" rəqəmlərlə öyrədirik

MNIST-də bütün rəqəmlər səliqə ilə mərkəzdədir. Biz isə mausla çəkəcəyik: bir az əyri, bir az kənarda, müxtəlif ölçüdə.
Ona görə training zamanı şəkilləri **random fırladırıq, sürüşdürürük, böyüdüb-kiçildirik**. Hər epoch-da şəbəkə bir az fərqli rəqəmlər görür.

Buna **data augmentation** deyilir. Yalnız train-ə tətbiq edirik, test-i dürüst saxlayırıq.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

aug_train_data = datasets.MNIST(root="data", train=True, download=True, transform=train_transform)
aug_train_loader = DataLoader(aug_train_data, batch_size=64, shuffle=True)

# eyni rəqəm, 8 dəfə augmentation-dan keçir
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax in axes:
    ax.imshow(aug_train_data[0][0].squeeze(), cmap="gray")
    ax.axis("off")
plt.show()

In [ ]:
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
history = []

for epoch in range(1, 6):
    loss = train_one_epoch(cnn, aug_train_loader, optimizer, loss_fn)
    acc = evaluate(cnn, test_loader)
    history.append((loss, acc))
    print(f"epoch {epoch}: loss={loss:.4f}  test accuracy={acc:.2%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot([h[0] for h in history], marker="o"); ax1.set_title("loss (train)"); ax1.set_xlabel("epoch")
ax2.plot([h[1] for h in history], marker="o", color="green"); ax2.set_title("accuracy (test)"); ax2.set_xlabel("epoch")
plt.show()

show_mistakes(cnn)

Təxminən **99%**. Səhvlər xeyli azaldı, qalanları da həqiqətən mübahisəlidir.

### Birinci layer-in filterləri nə "görür"
Hər kvadratcıq 32 filterdən biridir, 3×3 ölçüdə. Onları şəbəkə training zamanı **özü** tapıb:

In [ ]:
filters = cnn.conv1.weight.detach().cpu()      # (32, 1, 3, 3)
fig, axes = plt.subplots(4, 8, figsize=(8, 4))
for ax, f in zip(axes.flat, filters):
    ax.imshow(f.squeeze(), cmap="bwr")
    ax.axis("off")
plt.show()

### 💾 Modeli yadda saxlayırıq

Yalnız **çəkiləri** (`state_dict`) saxlayırıq. Bu, "layer-in adı → tensor" lüğətidir. Sonra yükləmək üçün
**eyni** `Net` class-ı lazımdır, ona görə də o, `draw_app.py` faylına olduğu kimi köçürülüb.

In [ ]:
torch.save(cnn.state_dict(), "mnist_cnn.pt")
print("mnist_cnn.pt faylında saxlandı")

# yoxlayaq: yeni şəbəkəyə yükləyirik
loaded = Net()
loaded.load_state_dict(torch.load("mnist_cnn.pt", map_location="cpu"))
loaded.eval()
print("yüklənmiş modelin accuracy-si:", evaluate(loaded.to(device), test_loader))

---
## 8️⃣ Proqram: rəqəmi sən çəkirsən, şəbəkə tapır

**`draw_app.py`** faylı bu notebook-un yanındadır. Terminaldan işə salırıq:

```bash
python draw_app.py
```

Pəncərə **Qt (PySide6)** ilə qurulub, macOS, Windows və Linux-da normal işləyir.

- mausun sol düyməsi ilə çəkirik
- şəbəkə elə çəkdiyin anda təxmin edir
- **"Təmizlə"** düyməsi və ya `C` / `Space` — silmək

### Ən vacib hissə: şəkli hazırlamaq
Şəbəkə MNIST şəkilləri üzərində öyrənib, ona görə bizim rəsm də **MNIST-ə oxşamalıdır**, yoxsa accuracy düşəcək:

1. 280×280 kətan, **qara fonda ağ** çəkirik (MNIST-dəki kimi)
2. rəqəmin ətrafından çərçivə ilə kəsirik
3. rəqəm **20×20**-yə sığsın deyə kiçildirik (proporsiyanı saxlayaraq)
4. **28×28** şəklin mərkəzinə qoyuruq (ətrafda 4 piksel boş yer qalır, MNIST-dəki kimi)
5. rəqəmin "ağırlıq mərkəzi" şəklin mərkəzinə düşsün deyə sürüşdürürük
6. eyni `MEAN` və `STD` ilə `Normalize`

Pəncərənin sağ tərəfində şəbəkənin əslində gördüyü 28×28 şəkil göstərilir.

### ✍️ Tapşırıqlar
1. Rəqəmləri **çox kiçik** və ya **küncdə** çəkin. Tapır? Niyə (ipucu: 2–5-ci addımlar)?
2. `draw_app.py`-da `Net` əvəzinə `MLP` modelini işlədin (onun çəkilərini başqa fayla saxlayın). Hansı şəbəkə **sizin** rəsmlərinizi daha yaxşı tanıyır?
3. Augmentation-ı silin və CNN-i yenidən öyrədin. Test accuracy demək olar ki, dəyişməyəcək. Bəs sizin rəsmlərdə?
4. ⭐ `Net`-ə üçüncü konvolyusiya layer-i əlavə edin və ya filterlərin sayını artırın. Daha yaxşı oldu?